## Establish Cassandra Connection

Connect to the local Cassandra cluster and select the `telecom_usage` keyspace for query operations.

In [2]:
from cassandra.cluster import Cluster

cluster = Cluster(["127.0.0.1"])
session = cluster.connect("telecom_usage")

print("Connected Keyspace:", session.keyspace)

Connected Keyspace: telecom_usage


## Load Source Dataset

Load the previously validated source dataset to support comparison and verification of Cassandra query results.

In [4]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/raw/MPD_sample_synthetic_ken_3000subs.csv")

df = pd.read_csv(DATA_PATH)

df["datetime"] = pd.to_datetime(
    df["datetime"],
    format="%Y-%m-%d %H:%M:%S"
)

df["event_date"] = df["datetime"].dt.date

print("Dataset loaded:", df.shape)

Dataset loaded: (1072608, 8)


## Query 1 – Retrieve Subscriber Events

Retrieve the five most recent events for a specific subscriber on a selected date.

In [6]:
sample = df.iloc[100]

msisdn = sample["msisdn"]
event_date = sample["event_date"]

subscriber_events = session.execute(
    """
    SELECT *
    FROM events_by_subscriber_day
    WHERE msisdn = %s
      AND event_date = %s
    LIMIT 5
    """,
    (msisdn, event_date)
)

for event in subscriber_events:
    print(event)

Row(msisdn='subscribers_00323', event_date=Date(20488), event_time=datetime.datetime(2026, 2, 4, 23, 45), cell_id=32385, data_type='CDR', latitude=-1.2654, longitude=36.9054, service='2G')
Row(msisdn='subscribers_00323', event_date=Date(20488), event_time=datetime.datetime(2026, 2, 4, 21, 37), cell_id=33597, data_type='CDR', latitude=-1.2674, longitude=36.9147, service='2G')
Row(msisdn='subscribers_00323', event_date=Date(20488), event_time=datetime.datetime(2026, 2, 4, 20, 38), cell_id=33597, data_type='CDR', latitude=-1.2674, longitude=36.9147, service='2G')
Row(msisdn='subscribers_00323', event_date=Date(20488), event_time=datetime.datetime(2026, 2, 4, 20, 29), cell_id=33597, data_type='CDR', latitude=-1.2674, longitude=36.9147, service='2G')
Row(msisdn='subscribers_00323', event_date=Date(20488), event_time=datetime.datetime(2026, 2, 4, 20, 27), cell_id=33597, data_type='CDR', latitude=-1.2674, longitude=36.9147, service='2G')


## Query 2 – Retrieve Cell Events

Retrieve the five most recent events recorded by a specific cell on a selected date.

In [7]:
sample = df.iloc[100]

cell_id = sample["cell_id"]
event_date = sample["event_date"]

cell_events = session.execute(
    """
    SELECT *
    FROM events_by_cell_day
    WHERE cell_id = %s
      AND event_date = %s
    LIMIT 5
    """,
    (cell_id, event_date)
)

for event in cell_events:
    print(event)

Row(cell_id=27968, event_date=Date(20488), event_time=datetime.datetime(2026, 2, 4, 16, 58), msisdn='subscribers_00323', data_type='CDR', latitude=-1.2987, longitude=36.8865, service='2G')
Row(cell_id=27968, event_date=Date(20488), event_time=datetime.datetime(2026, 2, 4, 16, 54), msisdn='subscribers_00323', data_type='CDR', latitude=-1.2987, longitude=36.8865, service='2G')
Row(cell_id=27968, event_date=Date(20488), event_time=datetime.datetime(2026, 2, 4, 15, 38), msisdn='subscribers_00323', data_type='CDR', latitude=-1.2987, longitude=36.8865, service='2G')
Row(cell_id=27968, event_date=Date(20488), event_time=datetime.datetime(2026, 2, 4, 14, 58), msisdn='subscribers_00323', data_type='CDR', latitude=-1.2987, longitude=36.8865, service='2G')
Row(cell_id=27968, event_date=Date(20488), event_time=datetime.datetime(2026, 2, 4, 14, 49), msisdn='subscribers_00323', data_type='CDR', latitude=-1.2987, longitude=36.8865, service='2G')


## Validate Subscriber Query Results

Compare the retrieved Cassandra events with the corresponding records in the original source dataset.

In [9]:
subscriber_events = list(session.execute(
    """
    SELECT *
    FROM events_by_subscriber_day
    WHERE msisdn = %s
      AND event_date = %s
    LIMIT 5
    """,
    (msisdn, event_date)
))

actual = pd.DataFrame([
    {
        "msisdn": event.msisdn,
        "event_time": event.event_time,
        "cell_id": event.cell_id,
        "data_type": event.data_type,
        "service": event.service
    }
    for event in subscriber_events
])

expected = (
    df[
        (df["msisdn"] == msisdn) &
        (df["event_date"] == event_date)
    ]
    .sort_values("datetime", ascending=False)
    .head(5)
    [["msisdn", "datetime", "cell_id", "data_type", "service"]]
    .rename(columns={"datetime": "event_time"})
    .reset_index(drop=True)
)

actual["event_time"] = pd.to_datetime(actual["event_time"])

print("Source records:", len(expected))
print("Cassandra records:", len(actual))
print("Results match:", expected.equals(actual))

Source records: 5
Cassandra records: 5
Results match: True


## Validate Cell Query Results

Compare the retrieved Cassandra cell events with the corresponding records in the original source dataset.

In [11]:
cell_events = list(session.execute(
    """
    SELECT *
    FROM events_by_cell_day
    WHERE cell_id = %s
      AND event_date = %s
    LIMIT 5
    """,
    (cell_id, event_date)
))

actual_cell = pd.DataFrame([
    {
        "cell_id": event.cell_id,
        "event_time": event.event_time,
        "msisdn": event.msisdn,
        "data_type": event.data_type,
        "service": event.service
    }
    for event in cell_events
])

expected_cell = (
    df[
        (df["cell_id"] == cell_id) &
        (df["event_date"] == event_date)
    ]
    .sort_values("datetime", ascending=False)
    .head(5)
    [["cell_id", "datetime", "msisdn", "data_type", "service"]]
    .rename(columns={"datetime": "event_time"})
    .reset_index(drop=True)
)

actual_cell["event_time"] = pd.to_datetime(actual_cell["event_time"])

print("Source records:", len(expected_cell))
print("Cassandra records:", len(actual_cell))
print("Results match:", expected_cell.equals(actual_cell))

Source records: 5
Cassandra records: 5
Results match: True


## Inspect Cassandra Partitions

Inspect the partition distribution of the subscriber events table using its composite partition key.

In [12]:
subscriber_partitions = (
    df.groupby(["msisdn", "event_date"])
    .size()
    .reset_index(name="event_count")
)

print("Total partitions:", len(subscriber_partitions))
print("\nSample partitions:")
display(subscriber_partitions.head(10))

Total partitions: 77711

Sample partitions:


,msisdn,event_date,event_count
0,subscribers_00000,2026-02-01,1
1,subscribers_00000,2026-02-04,1
2,subscribers_00000,2026-02-05,1
3,subscribers_00000,2026-02-06,5
4,subscribers_00000,2026-02-08,2
5,subscribers_00000,2026-02-09,2
6,subscribers_00000,2026-02-11,1
7,subscribers_00000,2026-02-12,1
8,subscribers_00000,2026-02-13,1
9,subscribers_00000,2026-02-14,2


In [13]:
largest_partition = subscriber_partitions.loc[
    subscriber_partitions["event_count"].idxmax()
]

partition_msisdn = largest_partition["msisdn"]
partition_date = largest_partition["event_date"]

partition_events = session.execute(
    """
    SELECT *
    FROM events_by_subscriber_day
    WHERE msisdn = %s
      AND event_date = %s
    """,
    (partition_msisdn, partition_date)
)

partition_events = list(partition_events)

print("Subscriber:", partition_msisdn)
print("Event Date:", partition_date)
print("Expected Events:", largest_partition["event_count"])
print("Cassandra Events:", len(partition_events))

display(pd.DataFrame(partition_events))

Subscriber: subscribers_02091
Event Date: 2026-02-13
Expected Events: 289
Cassandra Events: 289


,msisdn,event_date,event_time,cell_id,data_type,latitude,longitude,service
0,subscribers_02091,2026-02-13,2026-02-13 23:36:00,13724,CDR,-1.3106,36.814,2G
1,subscribers_02091,2026-02-13,2026-02-13 23:20:00,13724,CDR,-1.3106,36.814,2G
2,subscribers_02091,2026-02-13,2026-02-13 23:03:00,13724,CDR,-1.3106,36.814,2G
3,subscribers_02091,2026-02-13,2026-02-13 22:59:00,13724,CDR,-1.3106,36.814,2G
4,subscribers_02091,2026-02-13,2026-02-13 22:55:00,13724,CDR,-1.3106,36.814,2G
...,...,...,...,...,...,...,...,...
284,subscribers_02091,2026-02-13,2026-02-13 00:44:00,13724,CDR,-1.3106,36.814,2G
285,subscribers_02091,2026-02-13,2026-02-13 00:39:00,13724,CDR,-1.3106,36.814,2G
286,subscribers_02091,2026-02-13,2026-02-13 00:32:00,13724,CDR,-1.3106,36.814,2G
287,subscribers_02091,2026-02-13,2026-02-13 00:18:00,13724,CDR,-1.3106,36.814,2G


## Inspect Cell-Based Partitions

Analyze the partition distribution of the cell events table using its composite partition key.

In [14]:
cell_partitions = (
    df.groupby(["cell_id", "event_date"])
    .size()
    .reset_index(name="event_count")
)

print("Total partitions:", len(cell_partitions))
print("\nSample partitions:")
display(cell_partitions.head(10))

Total partitions: 342168

Sample partitions:


,cell_id,event_date,event_count
0,2,2026-02-15,1
1,2,2026-02-28,1
2,4,2026-02-14,1
3,8,2026-02-08,1
4,10,2026-02-13,1
5,11,2026-02-15,1
6,12,2026-02-07,1
7,13,2026-02-14,2
8,16,2026-02-14,2
9,19,2026-02-21,1


In [15]:
cluster.shutdown()

print("Cassandra connection closed successfully.")

Cassandra connection closed successfully.
